# XGBoost — third model (ratio 1:1, following the `tuning/v3` protocol)

## Tuning configuration
- **Dataset:** **1:1** pseudo-absences:presences ratio — the winning ratio identified
  in the sensitivity analysis of [`tuning/v3`](../../tuning/v3/README.md), rebuilt
  here with the SAME logic (same base table `pixel_year_full.csv`, same 3 km
  exclusion buffer, same `random_state=42` seed) so it is comparable figure by
  figure with the `1:1` rows for LR and RF already reported in `tuning/v3`.
- **Hyperparameter search:** `RandomizedSearchCV` with **150 configurations**
  randomly sampled (out of 300 possible combinations), `GroupKFold(10)` over 0.25°
  spatial blocks, `scoring='average_precision'`, **restricted to `year<=2019`**
  (same temporal-leakage correction as v2/v3).
- **`scale_pos_weight=1`** fixed (not tuned): with a 1:1 ratio the dataset is already
  balanced (same number of presences and pseudo-absences), so class weighting is not
  needed.

### Note on the requested grid
The original grid was specified in Random Forest terms (`n_estimators`, `max_depth`,
`max_features`, `min_samples_leaf`). XGBoost has neither `max_features` nor
`min_samples_leaf` — its native boosting equivalents are used, preserving the
same values where possible:

| Requested parameter (RF) | Requested values | XGBoost equivalent | Values used |
|---|---|---|---|
| `n_estimators` | 100, 150, 200, 250, 350 | `n_estimators` (same name and values) | 100, 150, 200, 250, 350 |
| `max_depth` | None, 10, 20, 30 | `max_depth` (0 = no limit in XGBoost) | 0, 10, 20, 30 |
| `max_features` | 'sqrt', 0.5, 4 | `colsample_bytree` (fraction of columns per tree) | √9/9≈0.333, 0.5, 4/9≈0.444 |
| `min_samples_leaf` | 4, 6, 10, 20, 30 | `min_child_weight` (minimum weight/samples per leaf) | 4, 6, 10, 20, 30 |

$5 \times 4 \times 3 \times 5 = 300$ possible combinations; 150 are sampled at random
(150 × 10 folds = 1,500 fits), a scale comparable to the 720 RF fits in v1/v3.

## What we expect from the model
XGBoost is an ensemble of sequential boosted trees (each tree corrects the errors
of the previous one), unlike Random Forest (independent trees in parallel). In the
fire-susceptibility literature it tends to match or slightly outperform RF in
PR-AUC when the dataset is small and the relationships between predictors are
non-linear, but it is also more prone to overfitting with little data (~4,154 rows
here) if not well regularized — that is why `max_depth`, `min_child_weight`, and
`colsample_bytree` are tuned in the search.

## What this notebook does
1. Rebuilds the 1:1 ratio dataset from `pixel_year_full.csv` (identical to `tuning/v3`).
2. Tunes XGBoost with `RandomizedSearchCV` (150 configurations, `year<=2019`).
3. Evaluates the tuned model with the full protocol (spatial block CV, 10 folds +
   temporal hold-out `>=2020`, with confusion matrices) — same as LR and RF.
4. Saves the results in this folder (`model/xgboost/`): `xgboost_metrics.csv` and
   `xgboost_vs_lr_rf_comparison.csv` (direct comparison against LR and RF at ratio 1:1).

In [1]:
# === train_xgboost.ipynb — Setup: 1:1 ratio dataset (winning ratio from tuning/v3) ===
# Rebuilds EXACTLY the same 1:1 dataset that tuning/v3/tune_v3.ipynb generated for
# Random Forest and Logistic Regression (same base table, same 3 km exclusion
# buffer, same random_state=42 seed), so XGBoost is directly comparable.
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              confusion_matrix)
from sklearn.base import clone

pixel_year = pd.read_csv('../../data/model_dataset/pixel_year_full.csv')
pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
BLOCK = 0.25
EXCL_BUFFER = 3000            # 3 km — identical to the one used in tuning/v3
buffer_deg = EXCL_BUFFER / 111000
RATIO = 1                     # winning ratio identified in tuning/v3 (see tuning/v3/README.md)
SEED = 42

presences = pixel_year[pixel_year['burned'] == 1].copy()
absences_all = pixel_year[pixel_year['burned'] == 0].copy()

kept_absences = []
for y in sorted(pixel_year['year'].unique()):
    pres_y = presences[presences['year'] == y][['lon','lat']].values
    abs_y  = absences_all[absences_all['year'] == y]
    if len(pres_y) == 0:
        kept_absences.append(abs_y); continue
    tree = cKDTree(pres_y)
    dists, _ = tree.query(abs_y[['lon','lat']].values, k=1)
    kept_absences.append(abs_y[dists > buffer_deg])
absences_far = pd.concat(kept_absences, ignore_index=True)

n_abs = min(len(absences_far), int(round(RATIO * len(presences))))
absences_sample = absences_far.sample(n=n_abs, random_state=SEED)
model_df = pd.concat([presences, absences_sample], ignore_index=True) \
             .sample(frac=1, random_state=SEED).reset_index(drop=True)
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                     (model_df['lat']//BLOCK).astype(int).astype(str)

n_pres = int(model_df['burned'].sum())
print("Dataset ratio 1:1 ->", model_df.shape,
      f"({n_pres} presences, {model_df.shape[0]-n_pres} pseudo-absences)")

Dataset ratio 1:1 -> (4154, 14) (2077 presencias, 2077 pseudo-ausencias)


In [2]:
# === RandomizedSearchCV — 150 random configurations, tuning ONLY with year<=2019 ===
tune_df = model_df[model_df['year'] <= 2019].copy()
X_tune = tune_df[pred_cols].values
y_tune = tune_df['burned'].astype(int).values
groups_tune = tune_df['block'].values
cv = GroupKFold(n_splits=10)

param_distributions_xgb = {
    'n_estimators':     [100, 150, 200, 250, 350],
    'max_depth':        [0, 10, 20, 30],                 # 0 = no limit (equivalent to None in RF)
    'colsample_bytree': [np.sqrt(len(pred_cols)) / len(pred_cols), 0.5, 4 / len(pred_cols)],
    'min_child_weight': [4, 6, 10, 20, 30],
}
n_combinations = (len(param_distributions_xgb['n_estimators']) *
                   len(param_distributions_xgb['max_depth']) *
                   len(param_distributions_xgb['colsample_bytree']) *
                   len(param_distributions_xgb['min_child_weight']))

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    scale_pos_weight=1,     # balanced 1:1 dataset -> no class correction needed
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_distributions_xgb,
    n_iter=150,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search.fit(X_tune, y_tune, groups=groups_tune)

print(f"Grid: {n_combinations} possible combinations -> 150 sampled x 10 folds = "
      f"{150 * 10} fits")
print("Best XGBoost hyperparameters:", xgb_search.best_params_)
print(f"Best PR-AUC (tuning CV, year<=2019): {xgb_search.best_score_:.3f}")

Grilla: 300 combinaciones posibles -> 150 muestreadas x 10 folds = 1500 fits
Mejores hiperparámetros XGBoost: {'n_estimators': 100, 'min_child_weight': 30, 'max_depth': 30, 'colsample_bytree': 0.4444444444444444}
Mejor PR-AUC (CV tuning, year<=2019): 0.867


In [3]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=BLOCK):
    """Validation protocol IDENTICAL to tuning/v3 (spatial block CV with 10 folds
    over the full dataset + temporal hold-out train<=2019/test>=2020), with
    confusion matrices — so XGBoost is directly comparable with LR and RF."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    cm_spatial = np.zeros((2, 2), dtype=int)
    for tr, te in gkf.split(X, y, groups):
        m = clone(estimator).fit(X[tr], y[tr])
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1 = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        cm_spatial += confusion_matrix(y[te], pred, labels=[0, 1])
    r = np.array(rows)
    print(f"  [{name}] SPATIAL  AUC={r[:,0].mean():.3f}±{r[:,0].std():.3f}  "
          f"PR-AUC={r[:,1].mean():.3f}±{r[:,1].std():.3f}  F1={r[:,2].mean():.3f}±{r[:,2].std():.3f}")

    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])
    cm_temporal = confusion_matrix(y[te], pred, labels=[0, 1])
    auc_t = roc_auc_score(y[te], prob)
    prauc_t = average_precision_score(y[te], prob)
    f1_t = f1_score(y[te], pred)
    print(f"  [{name}] TEMPORAL AUC={auc_t:.3f}  PR-AUC={prauc_t:.3f}  F1={f1_t:.3f}")

    return {
        'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0), 'spatial_cm': cm_spatial,
        'temporal': (auc_t, prauc_t, f1_t), 'temporal_cm': cm_temporal,
    }


def cm_to_dict(cm, prefix):
    tn, fp, fn, tp = cm.ravel()
    return {f'{prefix}_tn': int(tn), f'{prefix}_fp': int(fp),
            f'{prefix}_fn': int(fn), f'{prefix}_tp': int(tp)}


res_xgb = evaluate_model("XGBoost (tuned, ratio 1:1)", xgb_search.best_estimator_, model_df, pred_cols)

  [XGBoost (tuned, ratio 1:1)] SPATIAL  AUC=0.871±0.027  PR-AUC=0.864±0.034  F1=0.789±0.046
  [XGBoost (tuned, ratio 1:1)] TEMPORAL AUC=0.811  PR-AUC=0.717  F1=0.693


In [4]:
# === Save results — same format as tuning_v3_final_metrics.csv ===
row = {
    'ratio': '1:1',
    'model': 'XGBoost',
    'best_params': str(xgb_search.best_params_),
    'n_rows': model_df.shape[0],
    'n_presences': n_pres,
    'auc_spatial_mean': res_xgb['spatial'][0], 'auc_spatial_std': res_xgb['spatial_std'][0],
    'prauc_spatial_mean': res_xgb['spatial'][1], 'prauc_spatial_std': res_xgb['spatial_std'][1],
    'f1_spatial_mean': res_xgb['spatial'][2], 'f1_spatial_std': res_xgb['spatial_std'][2],
    'auc_temporal': res_xgb['temporal'][0], 'prauc_temporal': res_xgb['temporal'][1],
    'f1_temporal': res_xgb['temporal'][2],
}
row.update(cm_to_dict(res_xgb['spatial_cm'], 'cm_spatial'))
row.update(cm_to_dict(res_xgb['temporal_cm'], 'cm_temporal'))

xgb_metrics_df = pd.DataFrame([row])
xgb_metrics_df.to_csv('xgboost_metrics.csv', index=False)
print("Results saved to: model/xgboost/xgboost_metrics.csv")
xgb_metrics_df[['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
                'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']]

Resultados guardados en: model/xgboost/xgboost_metrics.csv


,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,1:1,XGBoost,4154,2077,0.87108,0.86377,0.788643,0.810905,0.716539,0.69258


In [5]:
# === Direct comparison against LR and RF (same 1:1 ratio, same validation protocol) ===
v3_sensitivity = pd.read_csv('../../tuning/v3/tuning_v3_sensitivity_metrics.csv')
cols = ['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
        'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']
lr_rf_11 = v3_sensitivity[v3_sensitivity['ratio'] == '1:1'][cols]

comparison_df = pd.concat([lr_rf_11, xgb_metrics_df[cols]], ignore_index=True)
comparison_df.to_csv('xgboost_vs_lr_rf_comparison.csv', index=False)
print("Comparison saved to: model/xgboost/xgboost_vs_lr_rf_comparison.csv\n")
comparison_df

Comparación guardada en: model/xgboost/xgboost_vs_lr_rf_comparison.csv



,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,1:1,Logistic Regression,4154,2077,0.845518,0.822546,0.753945,0.809537,0.746931,0.663230
1,1:1,Random Forest,4154,2077,0.878350,0.865608,0.789538,0.817119,0.703800,0.684303
2,1:1,XGBoost,4154,2077,0.871080,0.863770,0.788643,0.810905,0.716539,0.692580


## Conclusions

**Best hyperparameters (`RandomizedSearchCV`, 150/300 combinations, tuning `year<=2019`):**
`n_estimators=100`, `max_depth=30`, `colsample_bytree=0.444` (equivalent to `max_features=4`),
`min_child_weight=30`. Tuning PR-AUC (spatial CV, `year<=2019`): **0.867**.

### Results (ratio 1:1, dataset identical to LR/RF in `tuning/v3`)

| Model | Spatial AUC | Spatial PR-AUC | Spatial F1 | Temporal AUC | Temporal PR-AUC | Temporal F1 |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.846 | 0.823 | 0.754 | 0.810 | 0.747 | 0.663 |
| Random Forest | 0.878 | **0.866** | 0.790 | 0.817 | 0.704 | 0.684 |
| **XGBoost** | 0.871 | 0.864 | 0.789 | 0.811 | **0.717** | **0.693** |

(full table in [`xgboost_vs_lr_rf_comparison.csv`](xgboost_vs_lr_rf_comparison.csv))

### Does XGBoost improve the prediction?

- **Spatial validation:** XGBoost is practically tied with Random Forest (PR-AUC
  0.864 vs. 0.866, a difference of 0.002 — well below the standard deviation
  between folds for both models, ~0.03-0.04). There is no real improvement over RF
  when generalizing to new **locations**.
- **Temporal validation (train ≤2019 / test ≥2020):** XGBoost does improve over RF
  in PR-AUC (0.717 vs. 0.704, **+0.013**) and F1 (0.693 vs. 0.684, **+0.009**), although
  AUC-ROC is slightly lower (0.811 vs. 0.817). Since PR-AUC is the project's
  selection metric (rare event), this indicates that XGBoost generalizes better to
  new **years** than Random Forest, with the same dataset and validation protocol.
- Against Logistic Regression, XGBoost improves both validations by a wide margin
  (spatial PR-AUC +0.041, temporal PR-AUC +0.030), confirming that the non-linear
  model still adds value over the baseline.

### Conclusion for the 2026 susceptibility map

Since the 2026 map is a **projection to an unobserved future year** (a temporal
extrapolation, not a spatial one), the most relevant metric is the temporal
hold-out, not the spatial one. There, **XGBoost achieves the best PR-AUC and F1 of
the three models**, outperforming Random Forest on the validation that most closely
resembles the actual task of predicting 2026. The improvement is modest (+0.013
PR-AUC) and does not change the project's conclusion that the sampling ratio (1:1
vs. 2:1, see `tuning/v3`) matters far more than the choice of algorithm — but it
does suggest that **XGBoost, trained on the 1:1 dataset, is the best option
available for generating the 2026 susceptibility map**, with Random Forest as a
very close and more easily interpretable alternative.